        # 🔍 S4　支線：模型解釋——跟老闆說人話 2.0
        **統計冒險之旅 2026**　｜　支線（選修，隨時可做）　｜　支線任務　｜　🏅 100 XP

        📖 補充；資料：勇者咖啡會員
        　模型不只要準，還要能解釋

        ### 🎯 這一關你會學到
        - permutation importance：把一欄打亂看分數掉多少
- 部分相依圖：這個線索變大，預測怎麼變
- 單一會員的預測理由

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "S4"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["S4-1", "S4-2", "S4-3", "S4-4"]
_XP_EACH = 25
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""

class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_S4_1(run):
    out, ns = run()
    s = 抓變數(ns, "重要性")
    ok, msg = 資料框像(s, 列=12, 種類="Series")
    if not ok: return (False, msg)
    if not 約等於(s.max(), 0.08342, 0.03): return (False, "要用測試集、n_repeats=10、random_state=42、scoring='roc_auc'。")
    return (str(抓變數(ns, "最重要")) == "距上次來店天數", "最重要 = 重要性.index[0]。")
任務定義("S4-1", _check_S4_1, 提示="重要性.index[0]。")

def _check_S4_2(run):
    out, ns = run()
    d = 抓變數(ns, "部分相依", dict)
    if sorted(d) != [0, 7, 14, 30, 60, 90]: return (False, "部分相依 的鍵應為 0, 7, 14, 30, 60, 90。")
    if not 約等於(d[0], 0.60552, 0.05): return (False, "每個值 = rf.predict_proba(Xv)[:, 1].mean()。")
    return (str(抓變數(ns, "方向")) == "下降", "方向：比較 90 天與 0 天的平均機率。")
任務定義("S4-2", _check_S4_2, 提示="rf.predict_proba(Xv)[:, 1].mean()。")

def _check_S4_3(run):
    out, ns = run()
    s = 抓變數(ns, "貢獻")
    ok, msg = 資料框像(s, 列=12, 種類="Series")
    if not ok: return (False, msg)
    if not 約等於(s["來店次數"], 0.92580, 0.15): return (False, "貢獻 = lr.coef_[0] * z（z 是這位會員標準化後的值）。")
    return (str(抓變數(ns, "主因")) == "來店次數", "主因 = 貢獻.abs().idxmax()。")
任務定義("S4-3", _check_S4_3, 提示="貢獻.abs().idxmax()。")

def _check_S4_4(run):
    out, ns = run()
    s = str(抓變數(ns, "說人話")).strip()
    if len(s) < 40: return (False, "至少 40 個字，三句話。")
    if "距上次來店天數" not in s: return (False, "第一句要提到最重要的欄位「距上次來店天數」。")
    return ("回購" in s, "第三句的行動建議要提到「回購」。")
任務定義("S4-4", _check_S4_4, 提示="把 S4-1 的最重要欄位和 S4-2 的方向寫進去。")


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
members = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.2.1/data/coffee_members.csv")
y = members["回購"]
X = pd.get_dummies(members.drop(columns=["會員編號", "回購"]), drop_first=True).astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(200, random_state=42).fit(X_train, y_train)
print("測試 AUC：", round(roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]), 3))

## 🔍 支線：模型解釋——跟老闆說人話 2.0
老闆不會問「AUC 幾分」，老闆問三件事：
1. **模型主要靠什麼判斷？**（哪些欄位最重要）→ permutation importance
2. **這個欄位變大，預測會怎麼變？** → 部分相依（partial dependence）
3. **為什麼說這位客人會回購？** → 單一會員的預測理由

> ⚠️ L07 用的 `feature_importances_` 是「訓練時分裂用得多不多」，會偏愛數值範圍大的欄位。**permutation importance** 更誠實：把一欄打亂，看測試分數掉多少——掉越多越重要。

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

### 🎯 任務 S4-1　permutation importance

對測試集做 `permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc")`，把 `importances_mean` 做成以欄位為索引、由大到小排序的 Series `重要性`；`最重要` 是第一名的欄位名稱。

**預期結果（範例）**
```
最重要： 距上次來店天數
```

In [ ]:
# 🎯 任務 S4-1　permutation importance（請保留這一行）
結果 = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc")
重要性 = pd.Series(結果.importances_mean, index=X.columns).sort_values(ascending=False)
最重要 = ???
print(重要性.round(4).head(6)); print("最重要：", 最重要)
重要性.head(8)[::-1].plot(kind="barh", title="打亂這一欄，測試 AUC 掉多少"); plt.show()

In [ ]:
檢查("S4-1")   # ◀ 執行這一格，看看任務 S4-1 有沒有過關

## S4-2　部分相依：這個欄位變大，預測怎麼變？
做法很土但很有效：把測試集所有人的 `距上次來店天數` **全部改成同一個值**（0、7、14、30、60、90），看平均預測機率怎麼變。畫出來就是部分相依圖。

### 🎯 任務 S4-2　部分相依

對 `距上次來店天數` 在 `[0, 7, 14, 30, 60, 90]` 各做一次「整欄改成該值 → 平均預測機率」，存成字典 `部分相依`；`方向` 是 `"下降"` 或 `"上升"`（90 天的機率比 0 天低就是下降）。

**預期結果（範例）**
```
{0: 0.606, 7: 0.582, 14: 0.537, 30: 0.417, 60: 0.227, 90: 0.215} 下降
```

In [ ]:
# 🎯 任務 S4-2　部分相依（請保留這一行）
部分相依 = {}
for v in [0, 7, 14, 30, 60, 90]:
    Xv = X_test.copy(); Xv["距上次來店天數"] = v
    部分相依[v] = ???
方向 = "下降" if 部分相依[90] < 部分相依[0] else "上升"
print({k: round(p, 3) for k, p in 部分相依.items()}, 方向)
pd.Series(部分相依).plot(marker="o", title="距上次來店天數 的部分相依"); plt.ylabel("平均預測回購機率"); plt.show()

In [ ]:
檢查("S4-2")   # ◀ 執行這一格，看看任務 S4-2 有沒有過關

## S4-3　單一會員的預測理由
隨機森林很難拆成「每個欄位貢獻多少」，但邏輯斯迴歸可以：**標準化後的係數 × 這位會員的 z 值** 就是每個欄位對他的推力（正的往回購推、負的往不回購推）。

In [ ]:
sc = StandardScaler().fit(X_train)
lr = LogisticRegression(max_iter=1000).fit(sc.transform(X_train), y_train)
這位 = X_test.iloc[[0]]
print("會員編號：", members.loc[這位.index[0], "會員編號"], "｜ 隨機森林預測回購機率：", round(rf.predict_proba(這位)[0, 1], 3))

### 🎯 任務 S4-3　拆解一位會員

對測試集第一位會員（`X_test.iloc[[0]]`），算 `貢獻` = 邏輯斯係數 × 該會員標準化後的值（Series，索引為欄位名），`主因` 是絕對值最大的欄位名稱。

**預期結果（範例）**
```
主因： 來店次數
```

In [ ]:
# 🎯 任務 S4-3　拆解一位會員（請保留這一行）
sc = StandardScaler().fit(X_train)
lr = LogisticRegression(max_iter=1000).fit(sc.transform(X_train), y_train)
z = sc.transform(X_test.iloc[[0]])[0]
貢獻 = pd.Series(lr.coef_[0] * z, index=X.columns)
主因 = ???
print(貢獻.round(3).sort_values()); print("主因：", 主因)

In [ ]:
檢查("S4-3")   # ◀ 執行這一格，看看任務 S4-3 有沒有過關

### 🎯 任務 S4-4　跟老闆說人話

寫一段三句話的 `說人話`（字串，至少 40 字）：第一句說模型主要靠什麼判斷（要提到 `距上次來店天數`），第二句說這個欄位變大預測會怎麼變，第三句給一個行動建議（要提到「回購」）。

In [ ]:
# 🎯 任務 S4-4　跟老闆說人話（請保留這一行）
說人話 = """
???
"""
print(說人話.strip())

In [ ]:
檢查("S4-4")   # ◀ 執行這一格，看看任務 S4-4 有沒有過關

## 🌟 進階挑戰（不計分）
1. 用 `sklearn.inspection.PartialDependenceDisplay.from_estimator(rf, X_test, ["距上次來店天數"])` 畫官方版的部分相依圖，和你土法煉鋼的結果比一比。
2. 把 S4-3 換成測試集裡「預測機率最高」的那位會員，主因是什麼？

---
## 🔑 通關密語
　你已經能把模型翻譯成老闆聽得懂的三句話。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**支線完成！** 回入口網頁的「補給站 → 支線任務」蓋章。

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/